# Lee2019 Fine-Tuning with SignalJEPA Downstream Variants

Downstream fine-tuning on **Lee2019 Dataset**.

- Dataset: Lee2019 (MOABB)
- EEG channels only; bandpass 0.5-40 Hz; resample to 128 Hz
- 5-fold stratified within-subject cross-validation
- Model: configurable via `CONFIG["model_name"]`
- Pretrained weights: Hugging Face 16s-60 checkpoint
- Fine-tuning strategy: configurable via `CONFIG["strategy"]` (`new` or `full`)
- Training: Braindecode EEGClassifier with paper-aligned early stopping

# 1. Setup

## 1.1. Import Libraries

In [ ]:
import os
import random
import sys
from pathlib import Path
import platform
import json
import hashlib
from datetime import datetime
import math

import numpy as np
from numpy import multiply

import torch
from torch.utils.data import Subset

from braindecode import EEGClassifier
from braindecode.models import SignalJEPA_PreLocal, SignalJEPA_PostLocal, SignalJEPA_Contextual
from braindecode.datasets import MOABBDataset
from braindecode.datautil import load_concat_dataset
from braindecode.preprocessing import (
    Preprocessor,
    preprocess,
    create_windows_from_events,
)

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, roc_auc_score
from skorch.callbacks import EarlyStopping
from skorch.helper import predefined_split

import builtins

import mne
mne.set_log_level("WARNING")

print("All imports loaded successfully.")

## 1.2. Runtime & Path Validation

In [ ]:
print("Runtime Environment:")
print(f"  - Python: {sys.version}")
print(f"  - Platform: {platform.platform()}")

WORKING_DIR = Path.cwd().parent
print(f"\nWorking directory: {WORKING_DIR}")

# 2. Configuration

## 2.1. Config

In [ ]:
CONFIG = {
    # Paths
    "artifact_dir": str(WORKING_DIR / "artifacts" / "lee-2019-fine-tuning"),

    # Reproducibility
    "seed": 12,
    "set_seed": True,

    # Preprocessing
    "sfreq": 128,
    "bandpass_low": 0.5,
    "bandpass_high": 40.0,

    # Subjects (None = auto-select last 7 subjects, or specify a smaller list for debugging)
    "subjects_to_use": [51],

    # Paradigm settings
    "paradigm": "SSVEP", # 'SSVEP', 'MI', 'ERP'

    # Model settings
    "model_name": "SignalJEPA_PreLocal", # SignalJEPA_PreLocal, SignalJEPA_PostLocal, SignalJEPA_Contextual

    # Strategy settings
    "strategy": "new", # new, full
    "warmup_epochs": 10,

    # Cross-validation
    "cv_folds": 5,
    "val_split": 0.2,

    # Training
    "batch_size": 32,
    "n_epochs": 5000,
    "early_stopping_patience": 50,
    "learning_rate": 0.001,

    # Pretrained weights (legacy state-dict path)
    "pretrained_url": "https://huggingface.co/braindecode/SignalJEPA/resolve/main/signal-jepa_16s-60_adeuwv4s.pth",
}

# Output verbosity flags (set to True for more detailed output)
LOG_COMPACT = False
PRINT_MODEL_SUMMARY = True
PRINT_FREEZE_DETAILS = True
PRINT_FOLD_CLASS_COUNTS = True

In [ ]:
PARADIGM_CONFIGS = {
    "SSVEP": {
        "n_classes": 4,
        "base_trial_duration_s": 4.0,
        "target_window_duration_s": 4.2,
    },
    "MI": {
        "n_classes": 2,
        "base_trial_duration_s": 4.0,
        "target_window_duration_s": 4.2,
    },
    "ERP": {
        "n_classes": 2,
        "base_trial_duration_s": 1.0,
        "target_window_duration_s": 1.19,
    },
}

DEFAULT_SUBJECTS = [48, 49, 50, 51, 52, 53, 54]

## 2.2. Create Artifact Directory

In [ ]:
def create_run_id():
    # Generate unique run ID from timestamp + config hash.
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    config_str = json.dumps(CONFIG, sort_keys=True, default=str)
    config_hash = hashlib.md5(config_str.encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"

In [ ]:
RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / CONFIG["paradigm"] / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

## 2.3. Initialize Logger

In [ ]:
LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1)

def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop("sep", " ")
    end = kwargs.pop("end", "\n")
    flush = kwargs.pop("flush", False)
    file = kwargs.pop("file", None)

    message = sep.join(str(arg) for arg in args)

    # Preserve visual spacing for prints like print("\n...")
    leading_newlines = len(message) - len(message.lstrip("\n"))
    message_body = message[leading_newlines:]

    def _write_target(text):
        if file is None:
            sys.__stdout__.write(text) # type: ignore
            if flush:
                sys.__stdout__.flush() # type: ignore
        else:
            file.write(text)
            if flush and hasattr(file, "flush"):
                file.flush()

    # Apply leading blank lines first without timestamps
    if leading_newlines > 0:
        blanks = "\n" * leading_newlines
        _write_target(blanks)
        _LOG_FILE_HANDLE.write(blanks)

    if message_body:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        stamped = f"[{ts}] {message_body}"
        _write_target(stamped + end)
        _LOG_FILE_HANDLE.write(stamped + end)
    else:
        # If only newlines were printed, preserve end behavior without adding a timestamp
        _write_target(end)
        _LOG_FILE_HANDLE.write(end)

    if flush:
        _LOG_FILE_HANDLE.flush()

builtins.print = _timestamped_print

## 2.4. Device Configuration

In [ ]:
def resolve_device():
    """Resolve the compute device."""
    # prefer MPS > CUDA > CPU
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


DEVICE = resolve_device()
print(f"Using device: {DEVICE}")

## 2.5. Deterministic Seeding

In [ ]:
def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

    torch.use_deterministic_algorithms(True, warn_only=True)

def seed_worker(worker_id):
    _ = worker_id
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

BASE_SEED = int(CONFIG["seed"]) if CONFIG["seed"] is not None else None

if CONFIG["set_seed"]:
    if BASE_SEED is None:
        raise ValueError("Seed control is enabled but CONFIG['seed'] is None. Please specify a seed.")

    seed_everything(BASE_SEED)
    print(f"Seed control enabled: {CONFIG['set_seed']}")
    print(f"Base seed: {BASE_SEED}")
    print(f"Seed initialized: {BASE_SEED}")
else:
    print("Seed control disabled (CONFIG['set_seed'] = False)")

## 2.6. Save Configuration

In [ ]:
print("=" * 70)
print("CONFIGURATION")
print("=" * 70)
for key in sorted(CONFIG.keys()):
    print(f"  {key}: {CONFIG[key]}")
print("=" * 70)

config_path = ARTIFACT_DIR / "config.json"
with open(config_path, 'w') as f:
    json.dump(CONFIG, f, indent=2, default=str)
print(f"Config saved to: {config_path}")

# 3. Load and Prepare Data

## 3.1. Derived Constants

In [ ]:
def current_paradigm_config(paradigm, paradigm_configs):
    if paradigm not in ["SSVEP", "ERP", "MI"]:
        raise ValueError(f"Unsupported paradigm: {paradigm}. Must be one of 'SSVEP', 'ERP', 'MI'.")

    return (
        paradigm_configs[paradigm]["n_classes"],
        paradigm_configs[paradigm]["base_trial_duration_s"],
        paradigm_configs[paradigm]["target_window_duration_s"],
    )

In [ ]:
def compute_epoch_window(sfreq, target_trial_duration_s):
    """Compute the downstream window shape and fallback stop offset."""
    sfreq = float(sfreq)
    target_trial_duration_s = float(target_trial_duration_s)

    # Floor to avoid exceeding the requested duration.
    window_size_samples = int(math.floor(target_trial_duration_s * sfreq))

    print("Epoch window configuration:")
    print(f"  Target sfreq:                {sfreq:.0f} Hz")
    print(f"  Requested duration:          {target_trial_duration_s:.2f} s")
    print(f"  Expected window samples:     {window_size_samples}")

    return window_size_samples

In [ ]:
TARGET_N_CLASSES, BASE_TRIAL_DURATION_S, TARGET_TRIAL_DURATION_S = current_paradigm_config(
    CONFIG["paradigm"],
    PARADIGM_CONFIGS,
)

WINDOW_SAMPLES = compute_epoch_window(
    sfreq=CONFIG["sfreq"],
    target_trial_duration_s=TARGET_TRIAL_DURATION_S,
)

In [ ]:
PREPROCESSED_CONCAT_DATASETS_NAME = f"subjects_{'_'.join(str(s) for s in (CONFIG['subjects_to_use'] if CONFIG['subjects_to_use'] is not None else DEFAULT_SUBJECTS))}"
PREPROCESSED_CONCAT_DIR = WORKING_DIR / "preprocessed_concat_datasets" / PREPROCESSED_CONCAT_DATASETS_NAME

# Check if preprocessed concat dataset exists and set a variable accordingly
PREPROCESSED_CONCAT_DATASETS_EXISTS = PREPROCESSED_CONCAT_DIR.exists() and PREPROCESSED_CONCAT_DIR.is_dir()
print(f"\nPreprocessed concat dataset exists: {PREPROCESSED_CONCAT_DATASETS_EXISTS} (checked at: {PREPROCESSED_CONCAT_DIR})")

## 3.2. Load Data

In [ ]:
SUBJECTS = CONFIG["subjects_to_use"] if CONFIG["subjects_to_use"] is not None else DEFAULT_SUBJECTS

if not PREPROCESSED_CONCAT_DATASETS_EXISTS:
    DATASET_NAME = f"Lee2019_{CONFIG['paradigm']}"

    print(f"\nBuilding MOABBDataset for {DATASET_NAME}...")
    print(f"  Paradigm:       {CONFIG['paradigm']}")
    print(f"  Subject IDs:    {SUBJECTS}")

    DATASET = MOABBDataset(
        dataset_name=DATASET_NAME,
        subject_ids=SUBJECTS,
    )
    print(f"  Recordings loaded: {len(DATASET.datasets)}")
    print(f"  Labels: {sorted(set(DATASET.datasets[0].raw.annotations.description))}")

## 3.3. Preprocessing

In [ ]:
def scale_volts_to_microvolts(data):
    return multiply(data, 1e6)

def get_preprocessors(sfreq, bandpass_low, bandpass_high):
    """Build preprocessors with one explicit default path."""
    preprocessors = [
        Preprocessor("pick_types", eeg=True, meg=False, stim=False),
        Preprocessor("resample", sfreq=sfreq),
        Preprocessor("filter", l_freq=bandpass_low, h_freq=bandpass_high),
    ]

    return preprocessors

In [ ]:
if not PREPROCESSED_CONCAT_DATASETS_EXISTS:
    print("Applying Braindecode preprocessors to MOABBDataset...")
    preprocess(
        DATASET,
        get_preprocessors(
            sfreq=CONFIG["sfreq"],
            bandpass_low=CONFIG["bandpass_low"],
            bandpass_high=CONFIG["bandpass_high"],
        ),
        n_jobs=1
    )

    DATASET.save(PREPROCESSED_CONCAT_DIR, overwrite=True)
    print(f"Saved preprocessed raw concat dataset to: {PREPROCESSED_CONCAT_DIR}")

In [ ]:
LOADED_DATASET = load_concat_dataset(
    PREPROCESSED_CONCAT_DIR,
    preload=True,
    target_name=None,
    ids_to_load=None,
    )
print(f"Loaded raw concat dataset from: {PREPROCESSED_CONCAT_DIR}")

preprocess(
    LOADED_DATASET,
    [Preprocessor("set_eeg_reference", ref_channels="average")],
    n_jobs=1,
 )

CHS_INFO = LOADED_DATASET.datasets[0].raw.info["chs"]

In [ ]:
# Post-reload raw validation
subject_ids = sorted(set(str(ds.description["subject"]) for ds in LOADED_DATASET.datasets))
first_raw = LOADED_DATASET.datasets[0].raw

print("\nPost-reload raw validation")
print(f"  Recordings in LOADED_DATASET:   {len(LOADED_DATASET.datasets)}")
print(f"  Unique subject IDs:             {subject_ids}")
print(f"  sfreq (first rec):              {first_raw.info['sfreq']} Hz")
print(f"  EEG channel count:              {len(first_raw.ch_names)}")

## 3.4. Windowing

In [ ]:
def update_annotation_durations(dataset, target_duration_s):
    """Override event annotation durations in-place before window extraction."""
    updated_annotations = 0

    for ds in dataset.datasets:
        annotations = ds.raw.annotations
        if len(annotations) == 0:
            continue

        descriptions = np.asarray(annotations.description).astype(str)
        mask = np.ones(len(descriptions), dtype=bool)

        if not np.any(mask):
            continue

        durations = np.asarray(annotations.duration, dtype=float).copy()
        durations[mask] = float(target_duration_s)
        annotations.duration = durations
        updated_annotations += int(np.sum(mask))

    return updated_annotations

In [ ]:
def build_windows_dataset(
    dataset,
    target_duration_s,
 ):
    """Create event windows using annotation-duration as the default path."""
    update_annotation_durations(
        dataset,
        target_duration_s=target_duration_s,
    )

    windows_dataset = create_windows_from_events(
        dataset,
        trial_start_offset_samples=0,
        window_size_samples=None,
        window_stride_samples=None,
        drop_last_window=False,
        preload=True,
    )

    return windows_dataset

In [ ]:
WINDOWS_DATASET = build_windows_dataset(
    LOADED_DATASET,
    target_duration_s=TARGET_TRIAL_DURATION_S,
)

preprocess(
    WINDOWS_DATASET,
    [Preprocessor(scale_volts_to_microvolts)],
    n_jobs=1,
 )

# Window/sample diagnostics
sample0 = WINDOWS_DATASET[0]
x0 = sample0[0]
y0 = sample0[1]
print(f"Window shape: {tuple(x0.shape)}")
print(f"Window sample length expected={WINDOW_SAMPLES}, got={x0.shape[-1]}")

# Global label diagnostics
ALL_LABELS = np.asarray([int(WINDOWS_DATASET[i][1]) for i in range(len(WINDOWS_DATASET))], dtype=np.int64)
all_counts = np.bincount(ALL_LABELS, minlength=TARGET_N_CLASSES)
print(f"Global class counts: {all_counts.tolist()}")
print(f"Chance level ({TARGET_N_CLASSES}-class): {1.0 / TARGET_N_CLASSES:.2f}")

In [ ]:
def window_validation(windows_dataset):
    assert len(windows_dataset) > 0, "No windows created."
    sample0 = windows_dataset[0]
    x0 = sample0[0]
    assert x0.shape[-1] == WINDOW_SAMPLES, (
        f"Window length mismatch: got {x0.shape[-1]} expected {WINDOW_SAMPLES}"
    )
    all_labels = ALL_LABELS
    assert np.isin(all_labels, np.arange(TARGET_N_CLASSES)).all(), (
        f"Invalid labels found: {np.unique(all_labels)}"
    )

window_validation(WINDOWS_DATASET)

## 3.5. Get Subject Windows

In [ ]:
def get_subject_windows(windows_dataset, subject_ids):
    """Split windows by subject and ensure all requested subjects are available."""
    split_by_subject = windows_dataset.split("subject")
    subject_windows = {}

    for sid in subject_ids:
        key = str(sid)
        if key in split_by_subject:
            subject_windows[sid] = split_by_subject[key]
        elif sid in split_by_subject:
            subject_windows[sid] = split_by_subject[sid]
        else:
            raise RuntimeError(f"Subject {sid} not found in windows split keys: {list(split_by_subject.keys())[:10]}")

    return subject_windows

In [ ]:
SUBJECT_WINDOWS = get_subject_windows(WINDOWS_DATASET, SUBJECTS)

In [ ]:
def summarize_subject_windows(subject_windows, n_classes):
    """Summarize per-subject window counts and class balance."""
    print("Summarizing per-subject windows...")
    for subject_id in sorted(subject_windows):
        ds = subject_windows[subject_id]
        y = ALL_LABELS
        counts = np.bincount(y, minlength=n_classes)
        sample = ds[0]
        x = sample[0]

        print(
            f"  Subject {subject_id}: n_windows={len(ds)}, "
            f"window_shape={tuple(x.shape)}, class_counts={counts.tolist()}"
        )

summarize_subject_windows(SUBJECT_WINDOWS, TARGET_N_CLASSES)

# 4. Model

## 4.1. Build Model

In [ ]:
MODEL_REGISTRY = {
    "SignalJEPA_PreLocal": SignalJEPA_PreLocal,
    "SignalJEPA_PostLocal": SignalJEPA_PostLocal,
    "SignalJEPA_Contextual": SignalJEPA_Contextual,
}

def get_model_class(model_name):
    """Resolve downstream model class from config name with validation."""
    if model_name not in MODEL_REGISTRY:
        supported = ", ".join(MODEL_REGISTRY.keys())
        raise ValueError(f"Unsupported model_name '{model_name}'. Supported: {supported}")
    return MODEL_REGISTRY[model_name]

def build_model(model_name=None):
    """Instantiate the configured downstream SignalJEPA model."""
    selected_model_name = model_name or CONFIG["model_name"]
    model_cls = get_model_class(selected_model_name)

    # actual window duration
    trial_s = (WINDOW_SAMPLES / CONFIG["sfreq"] )
    if not LOG_COMPACT:
        print(f"Building model class: {selected_model_name}")
        print(f"Building model with input window duration: {trial_s:.4f} s")

    model = model_cls(
        sfreq=CONFIG["sfreq"],
        input_window_seconds=trial_s,
        chs_info=CHS_INFO,
        n_outputs=TARGET_N_CLASSES,
    )
    return model

In [ ]:
# Verify once that the selected model builds without error
test_model = build_model(CONFIG["model_name"])
total_p = sum(p.numel() for p in test_model.parameters())
trainable_p = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print(f"{CONFIG['model_name']} instantiated successfully.")
print(f"  Total parameters:     {total_p:,}")
print(f"  Trainable parameters: {trainable_p:,}")
if PRINT_MODEL_SUMMARY:
    print(test_model)
del test_model

## 4.2. Load Pretrained Weights

In [ ]:
def download_pretrained_state_dict(url):
    """Download the legacy Signal-JEPA checkpoint used for loading."""
    print("Initializing pretrained checkpoint download:")
    print(f"  {url}")
    ckpt = torch.hub.load_state_dict_from_url(url, map_location="cpu")
    if "student_backbone_state_dict" in ckpt:
        sd = ckpt["student_backbone_state_dict"]
    else:
        sd = ckpt
    print(f"  Downloaded {len(sd)} keys")
    return sd

In [ ]:
PRETRAINED_LOAD_RULES = {
    "SignalJEPA_PreLocal": {
        "load_prefixes": ("feature_encoder.",),
        "required_checkpoint_prefixes": ("feature_encoder.",),
        "expected_missing_prefixes": ("spatial_conv.", "final_layer."),
    },
    "SignalJEPA_PostLocal": {
        "load_prefixes": ("feature_encoder.",),
        "required_checkpoint_prefixes": ("feature_encoder.",),
        "expected_missing_prefixes": ("final_layer."),
    },
    "SignalJEPA_Contextual": {
        "load_prefixes": ("feature_encoder.", "transformer."),
        "required_checkpoint_prefixes": ("feature_encoder.", "transformer."),
        "expected_missing_prefixes": ("pos_encoder.", "final_layer."),
    },
}

PRETRAINED_STATE_DICT = None

def _starts_with_any_prefix(key, prefixes):
    return any(key.startswith(prefix) for prefix in prefixes)

def _validate_checkpoint_prefixes(model_name, state_dict, required_prefixes):
    for prefix in required_prefixes:
        if not any(k.startswith(prefix) for k in state_dict):
            raise RuntimeError(
                f"Checkpoint is inconsistent for {model_name}: "
                f"required prefix '{prefix}' not found."
            )

def get_pretrained_state_dict():
    global PRETRAINED_STATE_DICT
    if PRETRAINED_STATE_DICT is None:
        PRETRAINED_STATE_DICT = download_pretrained_state_dict(CONFIG["pretrained_url"])
    return PRETRAINED_STATE_DICT

def load_pretrained_weights_for_model(model, model_name, state_dict):
    """Load pretrained weights with model-specific key filtering and validation."""
    if model_name not in PRETRAINED_LOAD_RULES:
        raise ValueError(f"Unsupported model_name for loading: {model_name}")

    rules = PRETRAINED_LOAD_RULES[model_name]
    load_prefixes = rules["load_prefixes"]
    required_prefixes = rules["required_checkpoint_prefixes"]
    expected_missing_prefixes = rules["expected_missing_prefixes"]

    _validate_checkpoint_prefixes(model_name, state_dict, required_prefixes)

    filtered = {k: v for k, v in state_dict.items() if _starts_with_any_prefix(k, load_prefixes)}
    dropped = sorted([k for k in state_dict.keys() if k not in filtered])

    if len(filtered) == 0:
        raise RuntimeError(f"No keys selected for loading into {model_name}.")

    missing_keys, unexpected_keys = model.load_state_dict(filtered, strict=False)

    invalid_missing = [
        k for k in missing_keys
        if not _starts_with_any_prefix(k, expected_missing_prefixes)
    ]

    if unexpected_keys:
        raise RuntimeError(
            f"Unexpected keys while loading pretrained weights for {model_name}: "
            f"{unexpected_keys[:10]}"
        )

    if invalid_missing:
        raise RuntimeError(
            f"Unexpected missing keys for {model_name}: {invalid_missing[:10]}"
        )

    return {
        "loading_path": "legacy_state_dict",
        "model_name": model_name,
        "downloaded_keys": len(state_dict),
        "loaded_keys": len(filtered),
        "dropped_keys": len(dropped),
        "missing_keys": list(missing_keys),
        "unexpected_keys": list(unexpected_keys),
        "loaded_prefixes": list(load_prefixes),
        "expected_missing_prefixes": list(expected_missing_prefixes),
    }

def initialize_model_with_pretrained_weights(model_name):
    """Initialize the downstream model with legacy state-dict loading."""
    model = build_model(model_name)
    state_dict = get_pretrained_state_dict()
    summary = load_pretrained_weights_for_model(model, model_name, state_dict)
    return model, summary

## 4.3. Trainable Parameter Phases

In [ ]:
NEW_LAYER_PREFIXES = {
    "SignalJEPA_PreLocal": ("spatial_conv.", "final_layer."),
    "SignalJEPA_PostLocal": ("final_layer.",),
    "SignalJEPA_Contextual": ("pos_encoder.", "final_layer."),
}

def count_total_and_trainable_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

In [ ]:
def get_new_layer_prefixes(model_name):
    if model_name not in NEW_LAYER_PREFIXES:
        raise ValueError(f"Unsupported model_name for trainable phase logic: {model_name}")
    return NEW_LAYER_PREFIXES[model_name]

def set_trainable_params_for_phase(model, model_name, phase):
    """
    Configure trainable parameters by phase.

    phase='new' or 'warmup': only newly introduced downstream layers are trainable.
    phase='full': full model is trainable.
    """
    if phase not in ("new", "warmup", "full"):
        raise ValueError(f"Unsupported phase: {phase}")

    trainable_names = []

    if phase == "full":
        for _, param in model.named_parameters():
            param.requires_grad = True
        trainable_names = [name for name, p in model.named_parameters() if p.requires_grad]
        phase_groups = ["all_parameters"]
    else:
        downstream_prefixes = get_new_layer_prefixes(model_name)
        for _, param in model.named_parameters():
            param.requires_grad = False
        for name, param in model.named_parameters():
            if any(name.startswith(prefix) for prefix in downstream_prefixes):
                param.requires_grad = True
                trainable_names.append(name)
        phase_groups = list(downstream_prefixes)

    total, trainable = count_total_and_trainable_params(model)

    if trainable == 0:
        raise RuntimeError(
            f"No trainable parameters found for model={model_name}, phase={phase}."
        )

    return {
        "model_name": model_name,
        "phase": phase,
        "trainable_groups": phase_groups,
        "total_params": int(total),
        "trainable_params": int(trainable),
        "trainable_ratio": float(trainable / total),
        "trainable_names": trainable_names,
    }

def assert_expected_trainable_scope(summary, model_name, phase):
    if phase == "full":
        return

    allowed_prefixes = get_new_layer_prefixes(model_name)
    unexpected_names = [
        name for name in summary["trainable_names"]
        if not any(name.startswith(prefix) for prefix in allowed_prefixes)
    ]

    if unexpected_names:
        raise RuntimeError(
            f"Unexpected trainable parameters for {model_name} phase={phase}: {unexpected_names[:10]}"
        )

def describe_trainable_params(summary, max_names=12):
    print(f"      Trainable groups: {summary['trainable_groups']}")
    print(f"      Total params:     {summary['total_params']:,}")
    print(f"      Trainable params: {summary['trainable_params']:,}")
    print(f"      Trainable pct:    {100.0 * summary['trainable_ratio']:.2f}%")

    if PRINT_FREEZE_DETAILS or not LOG_COMPACT or len(summary["trainable_names"]) <= max_names:
        preview_names = summary["trainable_names"]
    else:
        preview_names = summary["trainable_names"][:max_names]

    print(f"      Trainable parameter names: {preview_names}")
    if len(preview_names) < len(summary["trainable_names"]):
        print(f"      ... {len(summary['trainable_names']) - len(preview_names)} additional trainable parameters hidden")

# 5. Training

## 5.1. EEGClassifier Fold Runner

In [ ]:
def get_targets(dataset):
    """Extract integer labels from a Braindecode dataset/subset."""
    return np.asarray([int(dataset[i][1]) for i in range(len(dataset))], dtype=np.int64)

In [ ]:
def build_classifier(model, valid_set, callbacks, max_epochs, fold_seed=None, warm_start=False):
    train_generator = None
    if fold_seed is not None:
        train_generator = torch.Generator()
        train_generator.manual_seed(fold_seed)

    clf_kwargs = {
        "batch_size": CONFIG["batch_size"],
        "max_epochs": int(max_epochs),
        "device": DEVICE,
        "callbacks": callbacks,
        "train_split": predefined_split(valid_set),
        "classes": range(TARGET_N_CLASSES),
        "iterator_train__shuffle": True,
        "iterator_train__num_workers": 0,
        "iterator_valid__num_workers": 0,
        "optimizer": torch.optim.Adam,
        "warm_start": warm_start,
    }

    if CONFIG["learning_rate"] is not None:
        clf_kwargs["lr"] = CONFIG["learning_rate"]

    if train_generator is not None:
        clf_kwargs["iterator_train__generator"] = train_generator

    return EEGClassifier(model, **clf_kwargs)

def run_one_batch_finite_sanity_check(model, train_set, model_name):
    """Fail fast if one-batch forward/loss contains NaN/Inf."""
    if len(train_set) == 0:
        raise RuntimeError(f"{model_name}: train_set is empty during sanity check.")

    batch_size = int(min(CONFIG["batch_size"], len(train_set)))
    sanity_loader = torch.utils.data.DataLoader(
        train_set,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    batch = next(iter(sanity_loader))

    if not isinstance(batch, (tuple, list)) or len(batch) < 2:
        raise RuntimeError(f"{model_name}: expected batch with at least (X, y), got type={type(batch)}")

    x_batch = batch[0]
    y_batch = batch[1]
    x_batch = torch.as_tensor(x_batch).float().to(DEVICE)
    y_batch = torch.as_tensor(y_batch).long().to(DEVICE)

    was_training = model.training
    model = model.to(DEVICE)
    model.eval()
    with torch.no_grad():
        logits = model(x_batch)
        if not torch.isfinite(logits).all():
            raise RuntimeError(
                f"{model_name}: non-finite logits detected in one-batch sanity check."
            )

        loss = torch.nn.functional.cross_entropy(logits, y_batch)
        if not torch.isfinite(loss):
            raise RuntimeError(
                f"{model_name}: non-finite loss detected in one-batch sanity check."
            )

    if was_training:
        model.train()

    print(
        f"    Sanity check passed: finite logits/loss on one training batch for {model_name}."
    )

def extract_binary_score_vector(score_output, expected_n_samples):
    """Normalize model score output into a 1D score vector for binary ROC-AUC."""
    if score_output is None:
        return None

    scores = np.asarray(score_output)

    if scores.ndim == 1:
        score_vec = scores.astype(float)
    elif scores.ndim == 2 and scores.shape[1] == 2:
        score_vec = scores[:, 1].astype(float)
    elif scores.ndim == 2 and scores.shape[1] == 1:
        score_vec = scores[:, 0].astype(float)
    else:
        print(
            f"    ROC-AUC score extraction skipped: unsupported score shape {tuple(scores.shape)} for binary task."
        )
        return None

    if score_vec.shape[0] != int(expected_n_samples):
        print(
            "    ROC-AUC score extraction skipped: sample count mismatch "
            f"(scores={score_vec.shape[0]} vs labels={expected_n_samples})."
        )
        return None

    if not np.isfinite(score_vec).all():
        print("    ROC-AUC score extraction skipped: non-finite values detected in score vector.")
        return None

    return score_vec

def compute_classification_metrics(y_true, y_pred, y_score=None, paradigm=None):
    """Compute accuracy/balanced accuracy always and ROC-AUC when applicable."""
    y_true = np.asarray(y_true).astype(int).reshape(-1)
    y_pred = np.asarray(y_pred).astype(int).reshape(-1)

    metrics = {
        "accuracy": None,
        "balanced_accuracy": None,
        "roc_auc": None,
    }

    try:
        metrics["accuracy"] = float(accuracy_score(y_true, y_pred)) # type: ignore
    except Exception as exc:
        print(f"    Accuracy computation failed: {exc}")

    try:
        metrics["balanced_accuracy"] = float(balanced_accuracy_score(y_true, y_pred)) # type: ignore
    except Exception as exc:
        print(f"    Balanced accuracy computation failed: {exc}")

    if paradigm == "ERP":
        if y_score is None:
            print("    ERP ROC-AUC skipped: score/probability output is unavailable.")
        elif len(np.unique(y_true)) < 2:
            print("    ERP ROC-AUC skipped: only one class present in y_true for this fold.")
        else:
            try:
                metrics["roc_auc"] = float(roc_auc_score(y_true, y_score)) # type: ignore
            except Exception as exc:
                print(f"    ERP ROC-AUC skipped: {exc}")

    return metrics

def run_single_fold(fold_id, subject_id, subject_dataset, idx_train, idx_val, idx_test):
    """Train and evaluate one fold using EEGClassifier on dataset subsets."""
    global PRETRAINED_CHECKPOINT_INFO

    if CONFIG["set_seed"]:
        fold_seed = BASE_SEED
        seed_everything(fold_seed) # type: ignore
    else:
        fold_seed = None

    train_set = Subset(subject_dataset, idx_train.tolist())
    valid_set = Subset(subject_dataset, idx_val.tolist())
    test_set = Subset(subject_dataset, idx_test.tolist())

    y_all = get_targets(subject_dataset)
    y_train = y_all[idx_train]
    y_val = y_all[idx_val]
    y_test = y_all[idx_test]

    train_counts = np.bincount(y_train, minlength=TARGET_N_CLASSES)
    val_counts = np.bincount(y_val, minlength=TARGET_N_CLASSES)
    test_counts = np.bincount(y_test, minlength=TARGET_N_CLASSES)

    if PRINT_FOLD_CLASS_COUNTS:
        print(f"    Train class counts: {train_counts.tolist()}")
        print(f"    Val class counts:   {val_counts.tolist()}")
        print(f"    Test class counts:  {test_counts.tolist()}")

    model_name = CONFIG["model_name"]
    strategy = CONFIG["strategy"]
    warmup_epochs = int(CONFIG["warmup_epochs"])

    model, pretrained_load_summary = initialize_model_with_pretrained_weights(model_name)
    PRETRAINED_CHECKPOINT_INFO = dict(pretrained_load_summary)

    print(f"    Downstream model:        {model_name}")
    print(f"    Fine-tune strategy:      {strategy}")
    print(f"    Warm-up active:          {strategy == 'full'}")
    print(f"    Warm-up epochs:          {warmup_epochs}")
    print(f"    Pretrained loading path: {pretrained_load_summary['loading_path']}")
    print(f"    Pretrained loading path: {pretrained_load_summary['loading_path']}")
    print(f"    Fallback triggered:      {pretrained_load_summary.get('fallback_triggered', False)}")
    print(f"    Missing keys:            {pretrained_load_summary.get('missing_keys', [])[:10]}")
    print(f"    Unexpected keys:         {pretrained_load_summary.get('unexpected_keys', [])[:10]}")

    if pretrained_load_summary.get("hub_error"):
        print(f"    Hub fallback reason:     {pretrained_load_summary['hub_error']}")

    if (
        model_name == "SignalJEPA_Contextual"
        and any(k.startswith("pos_encoder.") for k in pretrained_load_summary.get("missing_keys", []))
    ):
        print(
            "    Contextual note: missing pos_encoder keys are expected for this checkpoint "
            "configuration and will be trained as downstream-added parameters."
        )

    if strategy == "new":
        phase_1_summary = set_trainable_params_for_phase(model, model_name, "new")
        assert_expected_trainable_scope(phase_1_summary, model_name, "new")
        print("    Phase 1 (new):")
        describe_trainable_params(phase_1_summary)

        run_one_batch_finite_sanity_check(model, train_set, model_name)

        callbacks = [
            (
                "early_stopping",
                EarlyStopping(
                    monitor="valid_loss",
                    patience=CONFIG["early_stopping_patience"],
                    lower_is_better=True,
                    load_best=True,
                ),
            ),
        ]

        clf = build_classifier(
            model=model,
            valid_set=valid_set,
            callbacks=callbacks,
            max_epochs=int(CONFIG["n_epochs"]),
            fold_seed=fold_seed,
            warm_start=False,
        )

        phase_summaries = {
            "phase_1": phase_1_summary,
            "phase_2": None,
        }
        clf.fit(train_set, y=None)

    elif strategy == "full":
        if warmup_epochs < 1:
            raise ValueError("For strategy='full', warmup_epochs must be >= 1.")

        phase_1_summary = set_trainable_params_for_phase(model, model_name, "warmup")
        assert_expected_trainable_scope(phase_1_summary, model_name, "warmup")
        print("    Phase 1 (warmup):")
        describe_trainable_params(phase_1_summary)

        run_one_batch_finite_sanity_check(model, train_set, model_name)

        clf = build_classifier(
            model=model,
            valid_set=valid_set,
            callbacks=[],
            max_epochs=warmup_epochs,
            fold_seed=fold_seed,
            warm_start=True,
        )

        clf.fit(train_set, y=None)

        phase_2_summary = set_trainable_params_for_phase(clf.module_, model_name, "full")
        print("    Phase 2 (full):")
        describe_trainable_params(phase_2_summary)

        # Important: refresh optimizer so newly unfrozen parameters are trainable.
        clf.initialize_optimizer()

        remaining_epochs = int(CONFIG["n_epochs"]) - warmup_epochs
        if remaining_epochs < 1:
            raise ValueError(
                "CONFIG['n_epochs'] must be greater than CONFIG['warmup_epochs'] when strategy='full'."
            )

        callbacks = [
            (
                "early_stopping",
                EarlyStopping(
                    monitor="valid_loss",
                    patience=CONFIG["early_stopping_patience"],
                    lower_is_better=True,
                    load_best=True,
                ),
            ),
        ]

        clf.set_params(callbacks=callbacks, max_epochs=remaining_epochs)
        clf.fit(train_set, y=None)

        phase_summaries = {
            "phase_1": phase_1_summary,
            "phase_2": phase_2_summary,
        }
    else:
        raise ValueError(f"Unsupported strategy: {strategy}. Use 'new' or 'full'.")

    y_pred = clf.predict(test_set)

    y_score_raw = None
    if hasattr(clf, "predict_proba"):
        try:
            y_score_raw = clf.predict_proba(test_set)
        except Exception as exc:
            print(f"    predict_proba unavailable on this fold: {exc}")

    decision_fn = getattr(clf, "decision_function", None)
    if y_score_raw is None and callable(decision_fn):
        try:
            y_score_raw = decision_fn(test_set)
        except Exception as exc:
            print(f"    decision_function unavailable on this fold: {exc}")

    y_score = None
    if CONFIG["paradigm"] == "ERP":
        y_score = extract_binary_score_vector(y_score_raw, expected_n_samples=len(y_test))

    metrics = compute_classification_metrics(
        y_true=y_test,
        y_pred=y_pred,
        y_score=y_score,
        paradigm=CONFIG["paradigm"],
    )

    stopped_epoch = int(clf.history[-1]["epoch"]) if len(clf.history) > 0 else 0 # type: ignore
    valid_loss_curve = [
        (int(row["epoch"]), float(row["valid_loss"]))
        for row in clf.history
        if "valid_loss" in row
    ]

    if valid_loss_curve:
        best_epoch, best_valid_loss = min(valid_loss_curve, key=lambda x: x[1])
    else:
        best_epoch, best_valid_loss = None, None

    acc = metrics["accuracy"]
    bal_acc = metrics["balanced_accuracy"]
    roc_auc = metrics["roc_auc"]

    cm = confusion_matrix(y_test, y_pred, labels=np.arange(TARGET_N_CLASSES)).tolist()
    pred_hist = np.bincount(y_pred, minlength=TARGET_N_CLASSES).tolist()

    n_folds = CONFIG["cv_folds"]
    val_loss_str = f"{best_valid_loss:.4f}" if best_valid_loss is not None else "N/A"
    acc_str = f"{acc:.4f}" if acc is not None else "N/A"
    bal_acc_str = f"{bal_acc:.4f}" if bal_acc is not None else "N/A"
    roc_auc_str = f"{roc_auc:.4f}" if roc_auc is not None else "N/A"
    print(
        f"  Fold {fold_id}/{n_folds} | best_epoch={best_epoch} | stop={stopped_epoch} "
        f"| val_loss={val_loss_str} | acc={acc_str} | bal_acc={bal_acc_str} | roc_auc={roc_auc_str}"
    )

    return {
        "subject_id": str(subject_id),
        "fold_id": fold_id,
        "paradigm": CONFIG["paradigm"],
        "model_name": model_name,
        "strategy": strategy,
        "warmup_epochs": warmup_epochs,
        "n_train": len(train_set),
        "n_val": len(valid_set),
        "n_test": len(test_set),
        "train_class_counts": train_counts.tolist(),
        "val_class_counts": val_counts.tolist(),
        "test_class_counts": test_counts.tolist(),
        "pretrained_load": pretrained_load_summary,
        "phase_1_trainable_groups": phase_summaries["phase_1"]["trainable_groups"],
        "phase_1_trainable_params": phase_summaries["phase_1"]["trainable_params"],
        "phase_1_trainable_names": phase_summaries["phase_1"]["trainable_names"],
        "phase_2_trainable_groups": None if phase_summaries["phase_2"] is None else phase_summaries["phase_2"]["trainable_groups"],
        "phase_2_trainable_params": None if phase_summaries["phase_2"] is None else phase_summaries["phase_2"]["trainable_params"],
        "phase_2_trainable_names": None if phase_summaries["phase_2"] is None else phase_summaries["phase_2"]["trainable_names"],
        "best_epoch": best_epoch,
        "stopped_epoch": stopped_epoch,
        "best_valid_loss": best_valid_loss,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "roc_auc": roc_auc,
        "confusion_matrix": cm,
        "prediction_histogram": pred_hist,
    }


## 5.2. Subject Cross-Validation Runner

In [ ]:
def make_fold_splits(y, n_folds, val_split, n_classes):
    """Create stratified k-fold index splits for one subject."""
    indices = np.arange(len(y))
    skf = StratifiedKFold(
        n_splits=n_folds,
        shuffle=True,
        random_state=12,
    )
    folds = []

    for fold_id, (tv_idx, test_idx) in enumerate(skf.split(indices, y), start=1):
        y_tv = y[tv_idx]
        y_test_f = y[test_idx]

        split_seed = BASE_SEED if CONFIG["set_seed"] else None
        tr_local_idx, val_local_idx = train_test_split(
            np.arange(len(tv_idx)),
            test_size=val_split,
            stratify=y_tv,
            random_state=split_seed,
        )

        train_idx = tv_idx[tr_local_idx]
        val_idx = tv_idx[val_local_idx]

        if PRINT_FOLD_CLASS_COUNTS:
            train_counts = np.bincount(y[train_idx], minlength=n_classes)
            val_counts = np.bincount(y[val_idx], minlength=n_classes)
            test_counts = np.bincount(y_test_f, minlength=n_classes)
            print(f"  Fold {fold_id} class counts: train={train_counts.tolist()} val={val_counts.tolist()} test={test_counts.tolist()}")

        folds.append(
            {
                "fold_id": fold_id,
                "idx_train": train_idx,
                "idx_val": val_idx,
                "idx_test": test_idx,
            }
        )

    return folds

In [ ]:
def run_subject_cv(subject_id, subject_dataset, n_classes, cv_folds, val_split):
    """Run within-subject stratified CV for one subject."""
    y = get_targets(subject_dataset)
    counts = np.bincount(y, minlength=n_classes)
    print(f"\nSubject {subject_id}: {len(subject_dataset)} windows | class_counts={counts.tolist()}")

    folds = make_fold_splits(
        y,
        n_folds=cv_folds,
        val_split=val_split,
        n_classes=n_classes,
    )
    results = []

    for fold in folds:
        result = run_single_fold(
            fold["fold_id"],
            subject_id,
            subject_dataset,
            fold["idx_train"],
            fold["idx_val"],
            fold["idx_test"],
        )
        results.append(result)

    acc_values = [r["accuracy"] for r in results if r["accuracy"] is not None]
    bal_acc_values = [r["balanced_accuracy"] for r in results if r["balanced_accuracy"] is not None]
    roc_auc_values = [r["roc_auc"] for r in results if r["roc_auc"] is not None]

    mean_acc = float(np.mean(acc_values)) if len(acc_values) > 0 else None
    mean_bal_acc = float(np.mean(bal_acc_values)) if len(bal_acc_values) > 0 else None
    std_acc = float(np.std(acc_values)) if len(acc_values) > 0 else None
    std_bal_acc = float(np.std(bal_acc_values)) if len(bal_acc_values) > 0 else None
    mean_roc_auc = float(np.mean(roc_auc_values)) if len(roc_auc_values) > 0 else None
    std_roc_auc = float(np.std(roc_auc_values)) if len(roc_auc_values) > 0 else None


    acc_str = f"{mean_acc:.4f}±{std_acc:.4f}" if mean_acc is not None and std_acc is not None else "N/A"
    bal_acc_str = f"{mean_bal_acc:.4f}±{std_bal_acc:.4f}" if mean_bal_acc is not None and std_bal_acc is not None else "N/A"
    roc_auc_str = f"{mean_roc_auc:.4f}±{std_roc_auc:.4f}" if mean_roc_auc is not None and std_roc_auc is not None else "N/A"

    print(
        f"  Subject {subject_id} summary: acc={acc_str}  bal_acc={bal_acc_str}  "
        f"roc_auc={roc_auc_str}"
    )

    return results


## 5.3. Run All Subjects

In [ ]:
print("=" * 70)
print("STARTING CROSS-VALIDATION")
print("=" * 70)
print(f"Subjects:      {list(SUBJECT_WINDOWS.keys())}")
print(f"Paradigm:      {CONFIG['paradigm']}")
print(f"Model:         {CONFIG['model_name']}")
print(f"Strategy:      {CONFIG['strategy']}")
print(f"Warm-up epochs:{CONFIG['warmup_epochs']}")
print(f"CV folds:      {CONFIG['cv_folds']}")
print(f"Batch size:    {CONFIG['batch_size']}")
print(f"Learning rate: {CONFIG['learning_rate']}")
print(f"Max epochs:    {CONFIG['n_epochs']}")
print(f"Device:        {DEVICE}")
print(f"Seed control:  {CONFIG['set_seed']}")
print(f"CV seed:       {BASE_SEED}")
print("=" * 70)

FOLD_RESULTS = []

for sid, subject_ds in SUBJECT_WINDOWS.items():
    fold_results = run_subject_cv(
        sid,
        subject_ds,
        TARGET_N_CLASSES,
        CONFIG["cv_folds"],
        CONFIG["val_split"],
    )
    FOLD_RESULTS.extend(fold_results)

print(f"\nTotal folds completed: {len(FOLD_RESULTS)}")

# 6. Results

## 6.1. Aggregate Metrics

In [ ]:
# Per-subject aggregation
SUBJECT_METRICS = {}

for result in FOLD_RESULTS:
    sid = result["subject_id"]
    if sid not in SUBJECT_METRICS:
        SUBJECT_METRICS[sid] = {
            "accuracies": [],
            "balanced_accuracies": [],
            "roc_aucs": [],
        }
    SUBJECT_METRICS[sid]["accuracies"].append(result["accuracy"])
    SUBJECT_METRICS[sid]["balanced_accuracies"].append(result["balanced_accuracy"])
    SUBJECT_METRICS[sid]["roc_aucs"].append(result.get("roc_auc"))

for sid, m in SUBJECT_METRICS.items():
    acc_values = [v for v in m["accuracies"] if v is not None]
    bal_acc_values = [v for v in m["balanced_accuracies"] if v is not None]
    roc_auc_values = [v for v in m["roc_aucs"] if v is not None]

    m["mean_accuracy"] = float(np.mean(acc_values)) if len(acc_values) > 0 else None
    m["std_accuracy"] = float(np.std(acc_values)) if len(acc_values) > 0 else None
    m["mean_balanced_accuracy"] = float(np.mean(bal_acc_values)) if len(bal_acc_values) > 0 else None
    m["std_balanced_accuracy"] = float(np.std(bal_acc_values)) if len(bal_acc_values) > 0 else None
    m["mean_roc_auc"] = float(np.mean(roc_auc_values)) if len(roc_auc_values) > 0 else None
    m["std_roc_auc"] = float(np.std(roc_auc_values)) if len(roc_auc_values) > 0 else None

# Global aggregation
all_accs = [r["accuracy"] for r in FOLD_RESULTS if r["accuracy"] is not None]
all_bal_accs = [r["balanced_accuracy"] for r in FOLD_RESULTS if r["balanced_accuracy"] is not None]
all_roc_aucs = [r["roc_auc"] for r in FOLD_RESULTS if r.get("roc_auc") is not None]

mean_accuracy = float(np.mean(all_accs)) if len(all_accs) > 0 else None
std_accuracy = float(np.std(all_accs)) if len(all_accs) > 0 else None
mean_balanced_accuracy = float(np.mean(all_bal_accs)) if len(all_bal_accs) > 0 else None
std_balanced_accuracy = float(np.std(all_bal_accs)) if len(all_bal_accs) > 0 else None
mean_roc_auc = float(np.mean(all_roc_aucs)) if len(all_roc_aucs) > 0 else None
std_roc_auc = float(np.std(all_roc_aucs)) if len(all_roc_aucs) > 0 else None


GLOBAL_METRICS = {
    "mean_accuracy": mean_accuracy,
    "std_accuracy": std_accuracy,
    "mean_balanced_accuracy": mean_balanced_accuracy,
    "std_balanced_accuracy": std_balanced_accuracy,
    "mean_roc_auc": mean_roc_auc,
    "std_roc_auc": std_roc_auc,
    "n_subjects": len(SUBJECT_METRICS),
    "n_folds_total": len(FOLD_RESULTS),
}

print("=" * 70)
print("AGGREGATED RESULTS")
print("=" * 70)
for sid, m in sorted(SUBJECT_METRICS.items()):
    acc_str = f"{m['mean_accuracy']:.4f}±{m['std_accuracy']:.4f}" if m["mean_accuracy"] is not None and m["std_accuracy"] is not None else "N/A"
    bal_acc_str = f"{m['mean_balanced_accuracy']:.4f}±{m['std_balanced_accuracy']:.4f}" if m["mean_balanced_accuracy"] is not None and m["std_balanced_accuracy"] is not None else "N/A"
    roc_auc_str = f"{m['mean_roc_auc']:.4f}±{m['std_roc_auc']:.4f}" if m["mean_roc_auc"] is not None and m["std_roc_auc"] is not None else "N/A"
    print(
        f"  Subject {sid}:  acc={acc_str}  bal_acc={bal_acc_str}  "
        f"roc_auc={roc_auc_str} "
    )
print("-" * 70)
overall_acc_str = f"{GLOBAL_METRICS['mean_accuracy']:.4f}±{GLOBAL_METRICS['std_accuracy']:.4f}" if GLOBAL_METRICS["mean_accuracy"] is not None and GLOBAL_METRICS["std_accuracy"] is not None else "N/A"
overall_bal_acc_str = f"{GLOBAL_METRICS['mean_balanced_accuracy']:.4f}±{GLOBAL_METRICS['std_balanced_accuracy']:.4f}" if GLOBAL_METRICS["mean_balanced_accuracy"] is not None and GLOBAL_METRICS["std_balanced_accuracy"] is not None else "N/A"
overall_roc_auc_str = f"{GLOBAL_METRICS['mean_roc_auc']:.4f}±{GLOBAL_METRICS['std_roc_auc']:.4f}" if GLOBAL_METRICS["mean_roc_auc"] is not None and GLOBAL_METRICS["std_roc_auc"] is not None else "N/A"
print(
    f"  OVERALL:      acc={overall_acc_str}  bal_acc={overall_bal_acc_str}  "
    f"roc_auc={overall_roc_auc_str}"
)
print("=" * 70)

## 6.2. Save Outputs

In [ ]:
cv_results_path = ARTIFACT_DIR / "cv_results.json"
with open(cv_results_path, 'w') as f:
    json.dump(FOLD_RESULTS, f, indent=2)
print(f"CV results saved to:      {cv_results_path}")

subject_metrics_path = ARTIFACT_DIR / "subject_metrics.json"
with open(subject_metrics_path, 'w') as f:
    json.dump(SUBJECT_METRICS, f, indent=2)
print(f"Subject metrics saved to: {subject_metrics_path}")

global_metrics_path = ARTIFACT_DIR / "global_metrics.json"
with open(global_metrics_path, 'w') as f:
    json.dump(GLOBAL_METRICS, f, indent=2)
print(f"Global metrics saved to:  {global_metrics_path}")

run_metadata = {
    "run_id": RUN_ID,
    "artifact_dir": str(ARTIFACT_DIR),
    "paradigm": CONFIG["paradigm"],
    "subjects": [str(s) for s in SUBJECTS],
    "model_name": CONFIG["model_name"],
    "strategy": CONFIG["strategy"],
    "warmup_epochs": int(CONFIG["warmup_epochs"]),
    "pretrained_checkpoint_info": PRETRAINED_CHECKPOINT_INFO,
    "cv_seed": BASE_SEED,
    "global_metrics": GLOBAL_METRICS,
}

run_metadata_path = ARTIFACT_DIR / "run_metadata.json"
with open(run_metadata_path, 'w') as f:
    json.dump(run_metadata, f, indent=2)
print(f"Run metadata saved to:    {run_metadata_path}")

print(f"\nAll artifacts in: {ARTIFACT_DIR}")


## 6.3. Final Summary

In [ ]:
print("\n" + "=" * 70)
print("EXPERIMENT SUMMARY")
print("=" * 70)
print(f"Run ID:    {RUN_ID}")
print(f"Artifacts: {ARTIFACT_DIR}")
print()
print("Configuration:")
print(f"  Subjects:              {list(SUBJECTS)}")
print(f"  Paradigm:              {CONFIG['paradigm']}")
print(f"  Model:                 {CONFIG['model_name']}")
print(f"  Strategy:              {CONFIG['strategy']}")
print(f"  Warm-up epochs:        {CONFIG['warmup_epochs']}")
print(f"  Bandpass:              {CONFIG['bandpass_low']}-{CONFIG['bandpass_high']} Hz")
print(f"  Resample:              {CONFIG['sfreq']} Hz")
print(f"  CV folds:              {CONFIG['cv_folds']}")
print(f"  Validation split:      {CONFIG['val_split']}")
print(f"  Learning rate:         {CONFIG['learning_rate']}")
print(f"  Batch size:            {CONFIG['batch_size']}")
print(f"  Max epochs:            {CONFIG['n_epochs']}")
print(f"  CV seed:               {BASE_SEED}")
print(f"  Pretrained loading:    {PRETRAINED_CHECKPOINT_INFO.get('loading_path')}")
print()
print("Results:")
if GLOBAL_METRICS["mean_accuracy"] is not None and GLOBAL_METRICS["std_accuracy"] is not None:
    print(f"  Mean Accuracy:          {GLOBAL_METRICS['mean_accuracy']:.4f} ± {GLOBAL_METRICS['std_accuracy']:.4f}")
else:
    print("  Mean Accuracy:          N/A")

if GLOBAL_METRICS["mean_balanced_accuracy"] is not None and GLOBAL_METRICS["std_balanced_accuracy"] is not None:
    print(f"  Mean Balanced Accuracy: {GLOBAL_METRICS['mean_balanced_accuracy']:.4f} ± {GLOBAL_METRICS['std_balanced_accuracy']:.4f}")
else:
    print("  Mean Balanced Accuracy: N/A")

if GLOBAL_METRICS["mean_roc_auc"] is not None and GLOBAL_METRICS["std_roc_auc"] is not None:
    print(f"  Mean ROC-AUC:           {GLOBAL_METRICS['mean_roc_auc']:.4f} ± {GLOBAL_METRICS['std_roc_auc']:.4f}")
else:
    print("  Mean ROC-AUC:           N/A")